# Did the Strategy Work Today?

Run this **after 11:15 AM IST** (or after 3:30 PM for the full day).  
Everything is automatic — no manual input required.

**Timing clarity:**

| Time | What happens | Used for |
|------|-------------|----------|
| 9:15 | Market opens — NIFTY opening price known | Gap % calculation |
| 9:15 | All overnight indicators already known (US, SGX, DAX, VIX) | Signal check |
| 9:25 | Wait 10 min for noise to settle | Strike selection & entry |
| 9:26–11:15 | Monitor SL / Target hit | Exit logic |
| 11:15 | Hard exit regardless of SL/TP | Final P&L |

**Gap vs Strike — two different prices:**  
- Gap % = (NIFTY at **9:15 open** − yesterday close) / yesterday close  
- Strike = ATM based on NIFTY price **at 9:25** — not 9:15  
- NIFTY can move 50–150 pts in the first 10 min, so the 1-OTM strike at 9:25  
  may be completely different from the one at 9:15.  
- This notebook auto-fetches the 9:25 NIFTY price from Kite for correct strike selection.  
- You can override with  in Cell 4 if needed.


In [21]:
# ── Cell 1: Imports & Config ───────────────────────────────────────────────────
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import date, datetime, timedelta
from datetime import time as dtime
from pathlib import Path
from scipy.stats import norm as _norm
import warnings
warnings.filterwarnings('ignore')

BASE        = Path(globals().get('__vsc_ipynb_file__', __import__('pathlib').Path.cwd())).parent
SIGNALS_CSV = BASE / 'v2' / 'v2_reliable_signals.csv'

# Must match py_backtest.py and run_everyday.ipynb exactly
GAP_THRESHOLD        = 0.0015
GAP_LARGE_THRESHOLD  = 0.0050
VIX_RISING_THRESHOLD = 0.03
VIX_SPIKE_THRESHOLD  = 0.05
BASE_RATE            = 54.5
STOP_LOSS_PCT        = 0.15
PROFIT_TARGET_PCT    = 0.40
STRIKE_STEP          = 50
STRIKES_OTM          = 1
LOT_SIZE             = 75
BROKERAGE            = 80
RISK_FREE_RATE       = 0.065
W                    = 68

# ── Signal mode & lot caps — must match py_backtest.py ───────────────────────
SIGNAL_MODE   = "BEARISH_ONLY"   # "ALL" | "BEARISH_ONLY" | "BULLISH_ONLY"
DTE0_MAX_LOTS = 10   # lot cap on expiry day — wide spreads + extreme gamma risk
MAX_LOTS      = 20               # hard global cap on any single trade

print('Config loaded. Run Cell 2 to fetch today\'s data.')
print(f'  Signal mode : {SIGNAL_MODE}  |  DTE0 cap: {DTE0_MAX_LOTS}  |  Global cap: {MAX_LOTS}')

Config loaded. Run Cell 2 to fetch today's data.
  Signal mode : ALL  |  DTE0 cap: 10  |  Global cap: 20


In [37]:
# ── Cell 2: Fetch all data automatically ──────────────────────────────────────
today = date.today() - timedelta(days=4)

if today.weekday() >= 5:
    raise SystemExit(f'Today is {today.strftime("%A")} — market is closed.')

now_hour = datetime.now().hour
if now_hour < 15:
    print(f'WARNING: It is {datetime.now().strftime("%H:%M")} IST — market may not have closed yet.')
    print('Results will be incomplete. Run after 3:30 PM for full simulation.')
    print()

start_hist = str(today - timedelta(days=12))
end_hist   = str(today + timedelta(days=1))

print(f'Fetching data for {today} ...', end=' ', flush=True)

# ── Global overnight data (same as run_everyday) ──────────────────────────────
TICKERS = {
    'SP500'    : '^GSPC',
    'SGX'      : 'NKD=F',
    'DAX'      : '^GDAXI',
    'VIX_US'   : '^VIX',
    'NIFTY'    : '^NSEI',
    'VIX_INDIA': '^INDIAVIX',
}

raw = {}
for name, ticker in TICKERS.items():
    try:
        df = yf.download(ticker, start=start_hist, end=end_hist, progress=False, auto_adjust=True)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        raw[name] = df
    except Exception as e:
        raw[name] = pd.DataFrame()
        print(f'\n  WARNING: {name} fetch failed: {e}')

# ── Today's NIFTY 1-minute intraday data ──────────────────────────────────────
try:
    nifty_1m = yf.download('^NSEI', period='1d', interval='1m', progress=False, auto_adjust=True)
    if isinstance(nifty_1m.columns, pd.MultiIndex):
        nifty_1m.columns = nifty_1m.columns.get_level_values(0)
    if nifty_1m.index.tz is not None:
        nifty_1m.index = nifty_1m.index.tz_convert('Asia/Kolkata')
    else:
        nifty_1m.index = nifty_1m.index.tz_localize('Asia/Kolkata')
except Exception as e:
    nifty_1m = pd.DataFrame()
    print(f'\n  WARNING: intraday fetch failed: {e}')

print('done.')

# ── Extract values ─────────────────────────────────────────────────────────────
MAX_STALE = 5

def _ret(name, before=today):
    df = raw[name]
    rows = df[df.index.date < before]
    if len(rows) < 2: return None
    if (before - rows.index[-1].date()).days > MAX_STALE: return None
    return float((rows['Close'].iloc[-1] - rows['Close'].iloc[-2]) / rows['Close'].iloc[-2])

def _close(name, before=today):
    df = raw[name]
    rows = df[df.index.date < before]
    if len(rows) == 0: return None
    if (before - rows.index[-1].date()).days > MAX_STALE: return None
    return float(rows['Close'].iloc[-1])

def _today_close(name):
    df = raw[name]
    rows = df[df.index.date == today]
    if len(rows) == 0: return None
    return float(rows['Close'].iloc[-1])

sp500_ret    = _ret('SP500')
sgx_ret      = _ret('SGX')
dax_ret      = _ret('DAX')
vix_ret      = _ret('VIX_US')
nifty_ret    = _ret('NIFTY')
nifty_close  = _close('NIFTY')         # prev close
vix_india    = _today_close('VIX_INDIA') or _close('VIX_INDIA')  # today's India VIX

# Today's NIFTY open from 1-min data
nifty_open_today = None
nifty_close_today = None
first_hour_ret = None

if len(nifty_1m) > 0:
    open_candle = nifty_1m.between_time('09:20', '09:21')
    if len(open_candle) > 0:
        nifty_open_today = float(open_candle['Open'].iloc[0])
    close_candle = nifty_1m.between_time('11:14', '11:16')
    if len(close_candle) > 0:
        nifty_close_today = float(close_candle['Close'].iloc[-1])
    if nifty_open_today and nifty_close_today:
        first_hour_ret = (nifty_close_today - nifty_open_today) / nifty_open_today

# ── Print summary ──────────────────────────────────────────────────────────────
if nifty_close is None:
    raise SystemExit('NIFTY previous close unavailable. Possible market holiday.')

print(f'\n  NIFTY prev close   : {nifty_close:>10,.1f}')
print(f'  NIFTY today open   : {nifty_open_today:>10,.1f}' if nifty_open_today else '  NIFTY today open   :    MISSING')
print(f'  NIFTY 11:15 price  : {nifty_close_today:>10,.1f}' if nifty_close_today else '  NIFTY 11:15 price  :    MISSING')
print(f'  First hour return  : {first_hour_ret:>+10.2%}' if first_hour_ret is not None else '  First hour return  :    MISSING')
print(f'  India VIX          : {vix_india:>10.1f}' if vix_india else '  India VIX          :    MISSING')
print(f'  S&P 500 yesterday  : {sp500_ret:>+10.2%}' if sp500_ret is not None else '  S&P 500 yesterday  :    MISSING')
print(f'  SGX/Nikkei futures : {sgx_ret:>+10.2%}'   if sgx_ret   is not None else '  SGX/Nikkei futures :    MISSING')
print(f'  DAX yesterday      : {dax_ret:>+10.2%}'   if dax_ret   is not None else '  DAX yesterday      :    MISSING')
print(f'  VIX yesterday      : {vix_ret:>+10.2%}'   if vix_ret   is not None else '  VIX yesterday      :    MISSING')

Fetching data for 2026-03-31 ... 

done.

  NIFTY prev close   :   22,331.4
  NIFTY today open   :   22,226.3
  NIFTY 10:15 price  :   22,203.2
  First hour return  :     -0.10%
  India VIX          :       27.9
  S&P 500 yesterday  :     -0.39%
  SGX/Nikkei futures :     -0.53%
  DAX yesterday      :     +1.18%
  VIX yesterday      :     -1.42%


In [38]:
# ── Cell 3: Compute signals — gap is now KNOWN ─────────────────────────────────
if nifty_open_today is None:
    raise SystemExit('NIFTY open price unavailable — cannot determine gap or signal.')

gap = (nifty_open_today - nifty_close) / nifty_close

signals = {
    'Gap Up'          : gap          >  GAP_THRESHOLD,
    'Gap Up Strong'   : gap          >  GAP_LARGE_THRESHOLD,
    'Gap Down'        : gap          < -GAP_THRESHOLD,
    'Prev India UP'   : nifty_ret    is not None and nifty_ret  > 0,
    'Prev India DOWN' : nifty_ret    is not None and nifty_ret  < 0,
    'US UP'           : sp500_ret    is not None and sp500_ret  > 0,
    'US DOWN'         : sp500_ret    is not None and sp500_ret  < 0,
    'SGX UP'          : sgx_ret      is not None and sgx_ret    > 0,
    'SGX DOWN'        : sgx_ret      is not None and sgx_ret    < 0,
    'DAX UP'          : dax_ret      is not None and dax_ret    > 0,
    'VIX Rising'      : vix_ret      is not None and vix_ret    > VIX_RISING_THRESHOLD,
    'VIX Falling'     : vix_ret      is not None and vix_ret    < 0,
    'VIX Spike'       : vix_ret      is not None and vix_ret    > VIX_SPIKE_THRESHOLD,
}

# ── Human-readable value for each signal (shown in PASS/FAIL breakdown) ───────
def _signal_value(name):
    if name in ('Gap Up', 'Gap Up Strong', 'Gap Down'):
        return f'gap={gap:+.2%}'
    if name in ('Prev India UP', 'Prev India DOWN'):
        return f'prev India={nifty_ret:+.2%}' if nifty_ret is not None else 'MISSING'
    if name in ('US UP', 'US DOWN'):
        return f'SP500={sp500_ret:+.2%}' if sp500_ret is not None else 'MISSING'
    if name in ('SGX UP', 'SGX DOWN'):
        return f'SGX={sgx_ret:+.2%}' if sgx_ret is not None else 'MISSING'
    if name == 'DAX UP':
        return f'DAX={dax_ret:+.2%}' if dax_ret is not None else 'MISSING'
    if name in ('VIX Rising', 'VIX Falling', 'VIX Spike'):
        return f'VIX chg={vix_ret:+.2%}' if vix_ret is not None else 'MISSING'
    return ''

# Gap label
if gap > GAP_LARGE_THRESHOLD:  gap_label = f'GAP UP STRONG ({gap:+.2%})'
elif gap > GAP_THRESHOLD:      gap_label = f'GAP UP ({gap:+.2%})'
elif gap < -GAP_THRESHOLD:     gap_label = f'GAP DOWN ({gap:+.2%})'
else:                          gap_label = f'FLAT ({gap:+.2%})'

print(f'Gap    : {nifty_open_today:,.0f} vs prev close {nifty_close:,.0f}  ->  {gap_label}')
print(f'Active : {", ".join(k for k, v in signals.items() if v) or "(none)"}')
print()

# ── Load top-3 bearish + bullish ──────────────────────────────────────────────
if not SIGNALS_CSV.exists():
    raise FileNotFoundError(f'{SIGNALS_CSV} not found.')

reliable    = pd.read_csv(SIGNALS_CSV)
top_bearish = reliable[reliable['P_Down'] > BASE_RATE].sort_values('Edge_pp', ascending=False).head(10).reset_index(drop=True)
top_bullish = reliable[reliable['P_Down'] < (100 - BASE_RATE)].sort_values('P_Down').head(3).reset_index(drop=True)
bear_combos = list(top_bearish['Signal'])
bull_combos = list(top_bullish['Signal'])

def _fires(combo_str):
    return all(signals.get(s.strip(), False) for s in combo_str.split('+'))

def _print_combo_detail(combo_str, stats_row, direction):
    parts   = [s.strip() for s in combo_str.split('+')]
    passed  = [signals.get(p, False) for p in parts]
    all_ok  = all(passed)
    n_pass  = sum(passed)

    header  = '  ✓ FIRED' if all_ok else '  ✗      '
    if direction == 'bear':
        pct = f'P(DOWN)={stats_row["P_Down"]:.1f}%  Edge=+{stats_row["Edge_pp"]:.1f}%  N={int(stats_row["N"])}'
    else:
        pct = f'P(UP)={100-stats_row["P_Down"]:.1f}%  Edge=+{abs(stats_row["Edge_pp"]):.1f}%  N={int(stats_row["N"])}'
    print(f'{header}  {combo_str}  [{pct}]')

    for name, ok in zip(parts, passed):
        icon = '✓' if ok else '✗'
        val  = _signal_value(name)
        print(f'           {icon} {name:<22} {val}')

    if all_ok:
        print(f'           → ALL {len(parts)} signals satisfied — FIRES')
    else:
        print(f'           → {n_pass}/{len(parts)} signals satisfied — does not fire')
    print()

bear_fired = [c for c in bear_combos if _fires(c)] if SIGNAL_MODE != "BULLISH_ONLY" else []
bull_fired = [c for c in bull_combos if _fires(c)] if SIGNAL_MODE != "BEARISH_ONLY" else []

# ── Print combo breakdown ─────────────────────────────────────────────────────
print('─' * W)
if SIGNAL_MODE == 'BULLISH_ONLY':
    print('  BEARISH combos: [suppressed — BULLISH_ONLY mode]')
else:
    print('  BEARISH combos (buy PUT if any fires):')
    print()
    for c in bear_combos:
        row = top_bearish[top_bearish['Signal'] == c].iloc[0]
        _print_combo_detail(c, row, 'bear')

print('─' * W)
if SIGNAL_MODE == 'BEARISH_ONLY':
    print('  BULLISH combos: [suppressed — BEARISH_ONLY mode]')
else:
    print('  BULLISH combos (buy CALL if any fires):')
    print()
    for c in bull_combos:
        row = top_bullish[top_bullish['Signal'] == c].iloc[0]
        _print_combo_detail(c, row, 'bull')

# ── Final signal ──────────────────────────────────────────────────────────────
print('=' * W)
if bear_fired and bull_fired:
    action = 'CONFLICT'
    print(f'  SIGNAL: ⚡ CONFLICT — STAY OUT (both directions fired)')
elif bear_fired:
    action = 'BEARISH'
    r = top_bearish[top_bearish['Signal'] == bear_fired[0]].iloc[0]
    print(f'  SIGNAL: ▼ BEARISH — BUY PUT')
    print(f'  Combo  : {bear_fired[0]}')
    print(f'  P(DOWN)={r["P_Down"]:.1f}%  Edge=+{r["Edge_pp"]:.1f}%  N={int(r["N"])}')
elif bull_fired:
    action = 'BULLISH'
    r = top_bullish[top_bullish['Signal'] == bull_fired[0]].iloc[0]
    print(f'  SIGNAL: ▲ BULLISH — BUY CALL')
    print(f'  Combo  : {bull_fired[0]}')
    print(f'  P(UP)={100-r["P_Down"]:.1f}%  Edge=+{abs(r["Edge_pp"]):.1f}%  N={int(r["N"])}')
else:
    action = 'NO_SIGNAL'
    print(f'  SIGNAL: NEUTRAL — STAY OUT, no trade today')
    print(f'  No combo was fully satisfied across all its signals.')
print('=' * W)

Gap    : 22,226 vs prev close 22,331  ->  GAP DOWN (-0.47%)
Active : Gap Down, Prev India DOWN, US DOWN, SGX DOWN, DAX UP, VIX Falling

────────────────────────────────────────────────────────────────────
  BEARISH combos (buy PUT if any fires):

  ✗        Gap Up + Prev India DOWN + US UP + SGX UP  [P(DOWN)=74.6%  Edge=+20.0%  N=67]
           ✗ Gap Up                 gap=-0.47%
           ✓ Prev India DOWN        prev India=-2.14%
           ✗ US UP                  SP500=-0.39%
           ✗ SGX UP                 SGX=-0.53%
           → 1/4 signals satisfied — does not fire

  ✗        Gap Up + Prev India DOWN + SGX UP + DAX UP  [P(DOWN)=74.1%  Edge=+19.4%  N=54]
           ✗ Gap Up                 gap=-0.47%
           ✓ Prev India DOWN        prev India=-2.14%
           ✗ SGX UP                 SGX=-0.53%
           ✓ DAX UP                 DAX=+1.18%
           → 2/4 signals satisfied — does not fire

  ✗        Gap Up + Prev India DOWN + SGX UP + VIX Falling  [P(DOWN)=72.7%  Ed

In [24]:
# ── Cell 3b: Kite Connect setup (for real option prices) ──────────────────────
# Reads api_key from .env and access_token from v2/cache/access_token.txt
# If either is missing or login fails, Cell 4 falls back to Black-Scholes.

from kiteconnect import KiteConnect

kite = None

# Read api_key from .env (avoids hardcoding credentials)
_env = BASE / '.env'
API_KEY, API_SECRET = None, None
if _env.exists():
    for _l in _env.read_text().splitlines():
        if _l.strip().lower().startswith("api_key"):
            API_KEY    = _l.split("=", 1)[1].strip()
        elif _l.strip().lower().startswith("api_secret"):
            API_SECRET = _l.split("=", 1)[1].strip()
if not API_KEY or not API_SECRET:
    raise EnvironmentError(f"api_key/api_secret not found in {_env.resolve()}")

_token_file = BASE / 'v2' / 'cache' / 'access_token.txt'

# Cell 4 â€” Generate and print the Kite login URL
kite = KiteConnect(api_key=API_KEY)
login_url = kite.login_url()
print('Open this URL in your browser:\n')
print(login_url)
print('\nAfter login, copy the request_token from the redirected URL.')


Open this URL in your browser:

https://kite.zerodha.com/connect/login?api_key=vobx4ih6afnkqiaq&v=3

After login, copy the request_token from the redirected URL.


In [26]:
# Cell 5 â€” Generate session from request_token (run once per day after login)
REQUEST_TOKEN = 'sk7NJK6tcRaTKnvcDIIMLJFtWyItRHcH'   # <â”€â”€ paste here

session_data = kite.generate_session(REQUEST_TOKEN, api_secret=API_SECRET)
ACCESS_TOKEN  = session_data['access_token']
kite.set_access_token(ACCESS_TOKEN)

with open('kite_access_token.txt', 'w') as f:
    f.write(ACCESS_TOKEN)

print(f'Access token: {ACCESS_TOKEN}')
print('Saved to kite_access_token.txt')

Access token: 5Rz674qMr6G4Kz6RxO5DSQEHYi6YKLXw
Saved to kite_access_token.txt


In [ ]:
kite.profile()

{'user_id': 'UDF619',
 'user_type': 'individual/res_no_nn',
 'email': 'sayanchaudhuri14@gmail.com',
 'user_name': 'Sayan Chaudhuri',
 'user_shortname': 'Sayan',
 'broker': 'ZERODHA',
 'exchanges': ['BSE', 'NSE', 'MF'],
 'products': ['CNC', 'NRML', 'MIS', 'BO', 'CO'],
 'order_types': ['MARKET', 'LIMIT', 'SL', 'SL-M'],
 'avatar_url': None,
 'meta': {'demat_consent': 'consent'}}

In [31]:
# ── Cell 4: Simulate the trade using actual today's option data ───────────────
if action not in ('BEARISH', 'BULLISH'):
    print(f'No trade today ({action}). Nothing to simulate.')
else:
    # # ── Black-Scholes (kept only for tte() helper, NOT used for pricing) ──────
    # def bs_price(S, K, T, r, sigma, opt_type='CE'):
    #     if T <= 1e-7:
    #         return max(0.0, S-K) if opt_type=='CE' else max(0.0, K-S)
    #     sq = sigma * np.sqrt(T)
    #     d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / sq
    #     d2 = d1 - sq
    #     if opt_type == 'CE':
    #         return float(S*_norm.cdf(d1) - K*np.exp(-r*T)*_norm.cdf(d2))
    #     return float(K*np.exp(-r*T)*_norm.cdf(-d2) - S*_norm.cdf(-d1))

    def nearest_tuesday(d):
        """Return the nearest upcoming Tuesday (or today if already Tuesday)."""
        days = (1 - d.weekday()) % 7   # Monday=0, Tuesday=1
        return d + timedelta(days=days)

    def tte(t_obj):
        expiry_dt  = datetime.combine(expiry, dtime(15, 30))
        current_dt = datetime.combine(today, t_obj)
        secs = (expiry_dt - current_dt).total_seconds()
        return max(secs, 0.0) / (365.25 * 24 * 3600)

    # ── Trade setup ─────────────────────────────────────────────────────────────
    expiry   = nearest_tuesday(today)
    dte      = (expiry - today).days
    opt_type = 'PE' if action == 'BEARISH' else 'CE'

    # ── ATM at 9:25 (not 9:15) ───────────────────────────────────────────────
    # Gap % uses nifty_open_today (9:15 open) — that is correct.
    # But the STRIKE should be based on NIFTY price at 9:25, because NIFTY can
    # move 50-150 pts between 9:15 and 9:25. The 1-OTM strike you actually buy
    # at 9:25 may be completely different from the 9:15 ATM.
    #
    # Override: set NIFTY_AT_925_OVERRIDE to a specific price (e.g. 22450)
    # to force that price for strike calculation. Leave as None to auto-fetch.
    NIFTY_AT_925_OVERRIDE = None   # e.g. 22450 — or leave None for auto

    nifty_925 = NIFTY_AT_925_OVERRIDE
    if nifty_925 is None and kite is not None:
        try:
            nifty_token = 256265   # NSE:NIFTY 50 instrument token
            from_925    = datetime.combine(today, dtime(9, 25))
            to_925      = datetime.combine(today, dtime(9, 26))
            spot_candles = kite.historical_data(nifty_token, from_925, to_925, 'minute')
            if spot_candles:
                nifty_925 = float(spot_candles[0]['open'])
                print(f'NIFTY at 9:25  : {nifty_925:,.0f}  (fetched from Kite — used for strike)')
        except Exception as _e:
            print(f'Could not fetch NIFTY 9:25 price: {_e}')

    if nifty_925 is None:
        nifty_925 = nifty_open_today   # fallback to 9:15 open
        print(f'NIFTY at 9:25  : {nifty_925:,.0f}  (fallback: using 9:15 open — strike may differ)')

    atm    = round(nifty_925 / STRIKE_STEP) * STRIKE_STEP
    strike = atm - STRIKE_STEP*STRIKES_OTM if action == 'BEARISH' else atm + STRIKE_STEP*STRIKES_OTM
    print(f'ATM at 9:25    : {atm:,}  ->  1-OTM {opt_type}: {strike:,}')

    # ── Fetch real option data from Kite ──────────────────────────────────────
    option_1m      = None
    pricing_source = None   # None means no fallback — we'll abort if Kite fails

    if kite is None:
        raise RuntimeError('Kite connection is required. No BS fallback for live trades.')

    try:
        instruments = kite.instruments('NFO')
        inst_df     = pd.DataFrame(instruments)
        inst_df['expiry'] = pd.to_datetime(inst_df['expiry']).dt.date

        # ── Diagnostics: show what NIFTY expiries exist near our target ───────
        nifty_expiries = (
            inst_df[inst_df['name'] == 'NIFTY']['expiry']
            .drop_duplicates()
            .sort_values()
        )
        nearby = nifty_expiries[
            (nifty_expiries >= today) &
            (nifty_expiries <= today + timedelta(days=14))
        ].tolist()
        print(f'NIFTY expiries available (next 14 days): {nearby}')

        # ── If our computed expiry isn't in the exchange calendar, snap to nearest ──
        if expiry not in nifty_expiries.values and nearby:
            old_expiry = expiry
            expiry     = nearby[0]          # take the closest real expiry
            dte        = (expiry - today).days
            print(f'Expiry corrected: {old_expiry} → {expiry} (DTE={dte})')

        # ── Show strikes around ATM so we can confirm the strike exists ───────
        avail_strikes = (
            inst_df[
                (inst_df['name']            == 'NIFTY') &
                (inst_df['instrument_type'] == opt_type) &
                (inst_df['expiry']          == expiry)
            ]['strike']
            .sort_values()
            .values
        )
        atm_idx  = np.searchsorted(avail_strikes, atm)
        nearby_s = avail_strikes[max(0, atm_idx-3): atm_idx+4]
        print(f'Available {opt_type} strikes near ATM ({atm}): {nearby_s}')

        mask = (
            (inst_df['name']            == 'NIFTY')   &
            (inst_df['instrument_type'] == opt_type)  &
            (inst_df['expiry']          == expiry)    &
            (inst_df['strike']          == float(strike))
        )
        matched = inst_df[mask]

        if len(matched) == 0:
            raise RuntimeError(
                f'Instrument NOT found: NIFTY {expiry} {strike} {opt_type}. '
                f'Available strikes near ATM: {nearby_s}'
            )

        token    = int(matched.iloc[0]['instrument_token'])
        symbol   = matched.iloc[0]['tradingsymbol']
        from_dt  = datetime.combine(today, dtime(9, 25))
        to_dt    = datetime.combine(today, dtime(11, 16))
        records  = kite.historical_data(token, from_dt, to_dt, 'minute')

        if not records:
            raise RuntimeError(f'Kite returned 0 candles for {symbol}. Market may not have opened yet.')

        option_1m = pd.DataFrame(records)
        option_1m['date'] = pd.to_datetime(option_1m['date'])
        option_1m = option_1m.set_index('date')
        if option_1m.index.tzinfo is not None:
            option_1m.index = option_1m.index.tz_convert('Asia/Kolkata')
        else:
            option_1m.index = option_1m.index.tz_localize('Asia/Kolkata')

        pricing_source = f'Kite real data ({symbol}, {len(option_1m)} candles)'
        print(f'Option data     : {pricing_source}')

    except RuntimeError:
        raise   # surface the clear error message, don't swallow it
    except Exception as _e:
        raise RuntimeError(f'Kite option fetch failed: {_e}') from _e

    # ── Entry price (real data only) ──────────────────────────────────────────
    entry_price = float(option_1m.iloc[0]['open'])

    # ── Lot sizing with DTE caps ──────────────────────────────────────────────
    FIXED_LOTS   = 10
    cost_per_lot = entry_price * LOT_SIZE
    dte0_limited = min(FIXED_LOTS, DTE0_MAX_LOTS) if dte == 0 else FIXED_LOTS
    lots         = min(dte0_limited, MAX_LOTS)

    if entry_price < 0.5:
        raise RuntimeError(f'Option price {entry_price:.2f} < 0.5 pts — degenerate strike, no trade.')

    sl_price = entry_price * (1 - STOP_LOSS_PCT)
    tp_price = entry_price * (1 + PROFIT_TARGET_PCT)

    print(f'\nTrade setup:')
    print(f'  Instrument : NIFTY {today.strftime("%d%b%Y").upper()} {strike} {opt_type}')
    print(f'  Expiry     : {expiry}  (DTE={dte})')
    print(f'  Pricing    : {pricing_source}')
    print(f'  Entry      : {entry_price:.1f} pts  (at 9:25 AM open)')
    print(f'  Stop Loss  : {sl_price:.1f} pts  ({STOP_LOSS_PCT:.0%} below entry)')
    print(f'  Target     : {tp_price:.1f} pts  ({PROFIT_TARGET_PCT:.0%} above entry)')
    print()

    # ── Minute-by-minute simulation (real candles only) ───────────────────────
    exit_price  = None
    exit_reason = '11:15 exit'
    exit_time   = '11:15'

    # Uses high/low (not close) — matches backtest logic exactly.
    # A candle's low can touch SL even if close recovers; same for high/TP.
    first_hour_opt = option_1m.between_time('09:26', '11:15')
    for ts, row in first_hour_opt.iterrows():
        if float(row['low']) <= sl_price:
            exit_price  = sl_price
            exit_reason = 'Stop Loss hit'
            exit_time   = ts.strftime('%H:%M')
            break
        if float(row['high']) >= tp_price:
            exit_price  = tp_price
            exit_reason = 'Target hit'
            exit_time   = ts.strftime('%H:%M')
            break

    if exit_price is None:
        last_row   = option_1m.between_time('11:14', '11:16')
        exit_price = float(last_row.iloc[-1]['close']) if len(last_row) > 0 else entry_price

    pnl_pts = exit_price - entry_price
    pnl_rs  = pnl_pts * LOT_SIZE * lots - BROKERAGE * lots
    outcome = 'PROFIT' if pnl_rs > 0 else 'LOSS'

    # ── Direction correctness ─────────────────────────────────────────────────
    actual_dir = None
    if first_hour_ret is not None:
        actual_dir = 'DOWN' if first_hour_ret < 0 else 'UP'
    predicted  = 'DOWN' if action == 'BEARISH' else 'UP'
    correct    = (actual_dir == predicted) if actual_dir else None

    # ── Results ───────────────────────────────────────────────────────────────
    print('=' * W)
    print(f'  RESULT: {outcome}  ({exit_reason} at {exit_time})')
    print('=' * W)
    print(f'  Entry price : {entry_price:.1f} pts  (Rs {entry_price*LOT_SIZE:,.0f} per lot)')
    print(f'  Exit price  : {exit_price:.1f} pts  at {exit_time}')
    print()
    print(f'  Lots used   : {lots}  (cost/lot Rs {cost_per_lot:,.0f})')
    print(f'  P&L (pts)   : {pnl_pts:+.1f} pts  per lot')
    print(f'  P&L (Rs)    : Rs {pnl_rs:+,.0f}  ({lots} lot{"s" if lots > 1 else ""}, after brokerage)')
    print()
    if actual_dir:
        print(f'  Predicted   : {predicted}')
        print(f'  Actual      : {actual_dir}  ({first_hour_ret:+.2%} in first hour)')
        print(f'  Correct?    : {"YES" if correct else "NO"}')
    else:
        print(f'  Predicted   : {predicted}')
        print(f'  Actual dir  : unavailable (run after 11:15 AM)')
    print('=' * W)


NIFTY expiries available (next 14 days): [datetime.date(2026, 4, 7), datetime.date(2026, 4, 13)]
Available CE strikes near ATM (22250): [22100. 22150. 22200. 22250. 22300. 22350. 22400.]
Option data     : Kite real data (NIFTY2640722300CE, 57 candles)

Trade setup:
  Instrument : NIFTY 02APR2026 22300 CE
  Expiry     : 2026-04-07  (DTE=5)
  Pricing    : Kite real data (NIFTY2640722300CE, 57 candles)
  Entry      : 283.4 pts  (at 9:20 AM open)
  Stop Loss  : 226.8 pts  (20% below entry)
  Target     : 453.5 pts  (60% above entry)

  RESULT: LOSS  (10:15 exit at 10:15)
  Entry price : 283.4 pts  (Rs 21,259 per lot)
  Exit price  : 265.4 pts  at 10:15

  Lots used   : 5  (cost/lot Rs 21,259)
  P&L (pts)   : -18.0 pts  per lot
  P&L (Rs)    : Rs -7,150  (5 lots, after brokerage)

  Predicted   : UP
  Actual      : DOWN  (-0.10% in first hour)
  Correct?    : NO


In [32]:
from datetime import date, timedelta
import pandas as pd

today  = date.today()
nifty_open_today = 22450  # replace with actual or hardcode any value

STRIKE_STEP = 50
STRIKES_OTM = 1
opt_type    = 'CE'  # or 'PE'

def nearest_tuesday(d):
    days = (1 - d.weekday()) % 7
    return d + timedelta(days=days)

expiry = nearest_tuesday(today)
atm    = round(nifty_open_today / STRIKE_STEP) * STRIKE_STEP
strike = atm + STRIKE_STEP * STRIKES_OTM

instruments = kite.instruments('NFO')
inst_df = pd.DataFrame(instruments)
inst_df['expiry'] = pd.to_datetime(inst_df['expiry']).dt.date

nifty_opts = inst_df[(inst_df['name'] == 'NIFTY') & (inst_df['instrument_type'] == opt_type)]
print(f'Target  : NIFTY {expiry} {strike} {opt_type}')
print(f'Expiries: {sorted(nifty_opts["expiry"].unique())[:5]}')
print(f'Matched :\n{nifty_opts[nifty_opts["expiry"] == expiry][["tradingsymbol","strike","expiry"]].sort_values("strike")}')

Target  : NIFTY 2026-04-07 22500 CE
Expiries: [datetime.date(2026, 4, 7), datetime.date(2026, 4, 13), datetime.date(2026, 4, 21), datetime.date(2026, 4, 28), datetime.date(2026, 5, 5)]
Matched :
         tradingsymbol   strike      expiry
906  NIFTY2640720100CE  20100.0  2026-04-07
904  NIFTY2640720150CE  20150.0  2026-04-07
902  NIFTY2640720200CE  20200.0  2026-04-07
900  NIFTY2640720250CE  20250.0  2026-04-07
898  NIFTY2640720300CE  20300.0  2026-04-07
..                 ...      ...         ...
685  NIFTY2640726400CE  26400.0  2026-04-07
683  NIFTY2640726450CE  26450.0  2026-04-07
681  NIFTY2640726500CE  26500.0  2026-04-07
679  NIFTY2640726550CE  26550.0  2026-04-07
677  NIFTY2640726600CE  26600.0  2026-04-07

[131 rows x 3 columns]
